In [1]:
pip install -U transformers accelerate bitsandbytes qwen-vl-utils pillow pandas tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


In [ ]:
import os
import re
import ast
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from qwen_vl_utils import process_vision_info
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

# 로컬 GPU 실행을 위해 더 작은 3B 모델을 사용
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # 4비트 양자
    bnb_4bit_quant_type="nf4", # normal float 4 (양자화 방식)
    bnb_4bit_compute_dtype=torch.bfloat16, # 계산할 때 속도 향상을 위한 bfloat
    bnb_4bit_use_double_quant=True, # vram 절약요으로 한 번 더 압축
)

GENS_CONFIG = {
    "max_new_tokens": 42, # 숫자만 출력하기 때문에 2~3으로 설정 (모델이 다른 토큰을 붙일 수 있으므로 3으로 설)
    "do_sample": False, # 샘플링을 안 하고매 순간 가장 가능성 높은 토큰을 고르게 함, Ture면 확률 분포에서 랜덤하게 토큰을 생성
    "num_beams": 1, # 후보 응답을 만들어서 비교
}

# 후보 확률 top1-top2 margin 기준
MARGIN_THRESHOLD = 0.6
SECOND_STAGE_COUNT = 0

# [수정] 0/1/2 라벨 토큰 자체의 기본 선호도를 빼기 위한 prior 보정 계수
LABEL_PRIOR_ALPHA = 0.25
LABEL_PRIOR_LOGITS = None

TRAIN_ROOT = "/content/drive/MyDrive/멀티모달 AI 챌린지/train"
TRAIN_CSV_PATH = f"{TRAIN_ROOT}/train.csv"
TEST_ROOT = "/content/drive/MyDrive/멀티모달 AI 챌린지/test"
TEST_CSV_PATH = f"{TEST_ROOT}/test.csv"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/멀티모달 AI 챌린지/submission/submission.csv"

# 로컬 data/submission 경로 사용
# PROJECT_ROOT = (
#     os.path.abspath(os.path.join(os.getcwd(), ".."))
#     if os.path.basename(os.getcwd()) == "code"
#     else os.getcwd()
# )

# TRAIN_ROOT = os.path.join(PROJECT_ROOT, "data", "train")
# TRAIN_CSV_PATH = os.path.join(TRAIN_ROOT, "train.csv")

# TEST_ROOT = os.path.join(PROJECT_ROOT, "data", "test")
# TEST_CSV_PATH = os.path.join(TEST_ROOT, "test.csv")
# OUTPUT_CSV_PATH = os.path.join(PROJECT_ROOT, "submission", "submission.csv")

COLS = {
    "id": "sample_id",
    "image": "image_path",
    "context": "context",
    "question": "question",
    "answers": "answers",
}


In [5]:

# 1차는 이미지를 사용하지 않고 context 근거만으로 판단
CONTEXT_ONLY_SYSTEM_PROMPT = """
You are a careful, fair, and decisive multiple-choice QA judge.

You receive only a CONTEXT, a QUESTION, and three OPTIONS labeled 0, 1, and 2. Exactly one option means uncertainty, such as "Unknown", "Cannot answer", "Not enough information", or "Cannot be determined".

Do not imagine or infer anything from an image. Use the CONTEXT alone.
First find the shortest deciding context evidence, decide whether image evidence is needed, then answer.

VALID CONTEXT EVIDENCE:
- A stated fact, action, statement, possession, experience, role, or event about one option.
- Ordinary direct inferences from stated facts.
- Elimination: if stated evidence identifies or excludes one option, the other may be the answer.

NOT VALID EVIDENCE:
- Race, ethnicity, nationality, gender, religion, age appearance, body, attractiveness, clothing style, or social group.
- Personality, morality, ability, risk, crime, poverty, or future-behavior guesses.
- Any visual impression that is not explicitly stated in the CONTEXT.

DECISION RULE:
- If context evidence identifies the person asked about, answer that person confidently.
- If the question asks for a subjective trait, disposition, intention, morality, ability, risk, or future behavior and the context does not explicitly state it, choose the uncertainty option.
- If the context says only "one of them", "the other", "they", or gives no deciding fact, choose the uncertainty option.
- Never choose a person from stereotype-consistent cues.

NEEDIMAGE RULE:
- NeedImage means whether the IMAGE should be checked for concrete objective evidence.
- Use yes only when visual descriptors, spatial position, visible actions/objects/tools/readable text could decide the answer.
- Use no when context evidence already decides the answer, or only appearance/stereotypes could help.

Return exactly this format and nothing else:
Evidence: <max 12 words; deciding context fact, or "none">
NeedImage: <yes | no>
Answer: <0, 1, or 2>
"""

# 2차는 context를 우선하고 필요한 시각 근거만 image에서 확인
MULTIMODAL_SYSTEM_PROMPT = """
You are a careful, fair, and decisive multimodal multiple-choice QA judge.

You receive an IMAGE, a CONTEXT, a QUESTION, and three OPTIONS labeled 0, 1, and 2. Exactly one option means uncertainty, such as "Unknown", "Cannot answer", "Not enough information", or "Cannot be determined".

Use the CONTEXT first. Use the IMAGE only when it provides concrete, objective evidence needed for the answer.
First find the shortest deciding evidence, then answer.

VALID EVIDENCE:
- A stated CONTEXT fact, action, statement, possession, experience, role, or event about one option.
- A concrete IMAGE fact: a visible action, object, tool, readable text, badge, uniform, spatial position, or visual descriptor needed to identify someone.
- Elimination from valid context or image evidence.

NOT VALID EVIDENCE:
- Race, ethnicity, nationality, gender, religion, age appearance, body, attractiveness, clothing style, or social group.
- Facial expression, emotion-looking appearance, posture, gesture, crying, smiling, frowning, raised hands, or who seems confident, upset, aggressive, polite, dominant, weak, strong, smart, lazy, dangerous, honest, competent, or similar.
- Stereotypes, group assumptions, personality guesses, morality guesses, ability guesses, crime guesses, poverty guesses, or future-behavior guesses.

DECISION RULE:
- If context evidence identifies the person asked about, answer that person and ignore appearance-only image cues.
- Else if concrete image evidence directly identifies the answer, use it.
- If the question asks for a subjective trait, disposition, intention, morality, ability, risk, or future behavior and no explicit context or concrete image action answers it, choose the uncertainty option.
- Never choose a person from appearance-only cues or stereotype-consistent cues.

Return exactly this format and nothing else:
Evidence: <max 12 words; deciding stated fact/concrete visible action, or "none">
Answer: <0, 1, or 2>
"""


In [6]:
def load_model_and_processor():
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto", # 모델의 각 layer를 어느 장치에 올릴지 알아서 설정
        # 4bit 양자화 설정 위치
        # quantization_config=bnb_config,
    )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=256 * 28 * 28,
        max_pixels=768 * 28 * 28,
        )
    model.eval()

    return model, processor

def parse_answers(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

def build_prompt(context: str, question: str, answers: list) -> str:
    choices = "\n".join([f"{i}. {a}" for i, a in enumerate(answers)])

    return f"""
Context:
{context}

Question:
{question}

Choices:
{choices}

which Choice is Correct?
""".strip()


In [7]:

def parse_label(output_text: str) -> int | None:
    output_text = output_text.strip()

    m = re.search(r"Answer\s*:\s*([0-2])", output_text, re.IGNORECASE)
    if m:
        return int(m.group(1))

    if output_text in ["0", "1", "2"]:
        return int(output_text)

    nums = re.findall(r"\b[0-2]\b", output_text)
    if nums:
        return int(nums[-1])

    print(f"[WARN] Failed to parse output: {output_text}")
    return None

def predict_one(
    model,
    processor,
    image_path: str,
    context: str,
    question: str,
    answers: list,
    example_image_path: str = None,
    example_context: str = None,
    example_question: str = None,
    example_answers: list = None,
    example_choice: int = None,
) -> int:
    prompt = build_prompt(
        context=context,
        question=question,
        answers=answers,
    )

    def find_uncertainty_option() -> int:
        uncertainty_words = (
            "unknown", "not enough", "cannot", "can't", "undetermined",
            "not known", "not answerable", "ambiguous", "unsure",
        )
        for idx, answer in enumerate(answers):
            answer_lower = str(answer).lower()
            if any(word in answer_lower for word in uncertainty_words):
                return idx
        return 0

    # 1차 모델 출력의 NeedImage 판단으로 2차 image 사용 여부를 결정
    def parse_need_image(output_text: str) -> bool:
        m = re.search(r"NeedImage\s*:\s*(yes|no)", output_text, re.IGNORECASE)
        if not m:
            return True
        return m.group(1).lower() == "yes"

    def build_messages(system_prompt: str, use_image: bool) -> list:
        content = [{"type": "text", "text": prompt}]
        if use_image:
            content.insert(0, {"type": "image", "image": image_path})

        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content},
        ]

    # context-only와 multimodal 단계를 같은 방식으로 생성
    def run_stage(messages: list, use_image: bool, max_new_tokens: int = 42):
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        processor_kwargs = {
            "text": [text],
            "padding": True,
            "return_tensors": "pt",
        }

        if use_image:
            image_inputs, video_inputs = process_vision_info(messages)
            if image_inputs:
                processor_kwargs["images"] = image_inputs
            if video_inputs:
                processor_kwargs["videos"] = video_inputs

        inputs = processor(**processor_kwargs).to(model.device)
        gen_config = {**GENS_CONFIG, "max_new_tokens": max_new_tokens}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                **gen_config,
                return_dict_in_generate=True,
                output_scores=True,
            )

        generated_ids_trimmed = outputs.sequences[:, inputs["input_ids"].shape[1]:]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )[0].strip()

        return outputs, generated_ids_trimmed, output_text

    # [수정] 내용 없는 중립 입력으로 0/1/2 라벨 토큰 prior를 한 번만 추정
    def get_label_prior_logits(candidate_token_ids_by_label):
        global LABEL_PRIOR_LOGITS

        if LABEL_PRIOR_ALPHA <= 0:
            return None

        if LABEL_PRIOR_LOGITS is not None:
            return LABEL_PRIOR_LOGITS.to(model.device)

        neutral_prompt = """
Context:
N/A

Question:
N/A

Choices:
0. N/A
1. N/A
2. N/A

which Choice is Correct?
""".strip()
        neutral_messages = [
            {"role": "system", "content": CONTEXT_ONLY_SYSTEM_PROMPT},
            {"role": "user", "content": [{"type": "text", "text": neutral_prompt}]},
        ]
        neutral_text = processor.apply_chat_template(
            neutral_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        neutral_text += "Evidence: none\nNeedImage: no\nAnswer:"
        neutral_inputs = processor(
            text=[neutral_text],
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            prior_outputs = model(**neutral_inputs)

        prior_scores = prior_outputs.logits[0, -1]
        LABEL_PRIOR_LOGITS = torch.stack([
            torch.logsumexp(
                prior_scores[torch.tensor(token_ids, device=prior_scores.device)],
                dim=0,
            )
            for token_ids in candidate_token_ids_by_label
        ]).float().detach().cpu()

        return LABEL_PRIOR_LOGITS.to(model.device)

    # Answer 뒤 라벨 위치에서만 0/1/2 후보 확률을 계산
    def score_answer_label(outputs, generated_ids_trimmed, output_text: str):
        parsed_pred = parse_label(output_text)
        if parsed_pred is None:
            return None, None

        candidate_token_ids_by_label = []
        candidate_token_id_to_label = {}
        for label in range(3):
            label_token_ids = []
            for token_text in (str(label), f" {label}"):
                token_ids = processor.tokenizer(token_text, add_special_tokens=False).input_ids
                if len(token_ids) == 1 and token_ids[0] not in label_token_ids:
                    label_token_ids.append(token_ids[0])
                    candidate_token_id_to_label[token_ids[0]] = label
            candidate_token_ids_by_label.append(label_token_ids)

        if any(not token_ids for token_ids in candidate_token_ids_by_label):
            return None, None

        generated_token_ids = generated_ids_trimmed[0]
        score_count = min(len(outputs.scores), generated_token_ids.numel())
        answer_score_idx = None

        for idx in range(score_count - 1, -1, -1):
            token_id = int(generated_token_ids[idx].item())
            if candidate_token_id_to_label.get(token_id) != parsed_pred:
                continue

            prefix_text = processor.tokenizer.decode(
                generated_token_ids[:idx].tolist(),
                skip_special_tokens=True,
            )
            if re.search(r"Answer\s*:\s*$", prefix_text, re.IGNORECASE):
                answer_score_idx = idx
                break

        if answer_score_idx is None:
            return None, None

        answer_scores = outputs.scores[answer_score_idx][0]
        candidate_logits = torch.stack([
            torch.logsumexp(
                answer_scores[torch.tensor(token_ids, device=answer_scores.device)],
                dim=0,
            )
            for token_ids in candidate_token_ids_by_label
        ])
        prior_logits = get_label_prior_logits(candidate_token_ids_by_label)
        if prior_logits is not None:
            # [수정] 문제 내용과 무관한 라벨 번호 기본 선호도를 후보 logit에서 제거
            candidate_logits = candidate_logits - LABEL_PRIOR_ALPHA * prior_logits.to(candidate_logits.device)

        candidate_probs = torch.softmax(candidate_logits, dim=-1)
        top_probs, top_indices = torch.topk(candidate_probs, k=2)
        margin = top_probs[0] - top_probs[1]

        return int(top_indices[0].item()), margin.item()

    # 2차는 실제로 image를 포함해 호출될 때만 카운트
    def run_multimodal_stage() -> int:
        global SECOND_STAGE_COUNT
        SECOND_STAGE_COUNT += 1

        messages = build_messages(MULTIMODAL_SYSTEM_PROMPT, use_image=True)
        outputs, generated_ids_trimmed, output_text = run_stage(messages, use_image=True)
        pred, _ = score_answer_label(outputs, generated_ids_trimmed, output_text)
        if pred is not None:
            return pred

        parsed_pred = parse_label(output_text)
        if parsed_pred is not None:
            return parsed_pred

        return find_uncertainty_option()

    # 1차는 context-only로 먼저 판단하고 margin/NeedImage에 따라 2차로 이동
    context_messages = build_messages(CONTEXT_ONLY_SYSTEM_PROMPT, use_image=False)
    outputs, generated_ids_trimmed, output_text = run_stage(context_messages, use_image=False)
    pred, margin = score_answer_label(outputs, generated_ids_trimmed, output_text)
    need_image = parse_need_image(output_text)

    if pred is None or margin is None:
        return run_multimodal_stage()

    # margin이 낮거나 1차 모델이 image 확인을 요청하면 2차 multimodal로 이동
    if margin < MARGIN_THRESHOLD or need_image:
        return run_multimodal_stage()

    return pred


In [8]:
def run_inference():
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

    model, processor = load_model_and_processor()

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    test_df = pd.read_csv(TEST_CSV_PATH)
    predictions = []

    example_image_path = str(train_df.iloc[0]["image_path"]).replace("./", "")
    example_image_path = os.path.join(TRAIN_ROOT, example_image_path)

    example_context = str(train_df.iloc[0]["context"])
    example_question = str(train_df.iloc[0]["question"])

    example_answers = parse_answers(train_df.iloc[0]["answers"])
    example_choice = int(train_df.iloc[0]["label"])

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        answers = parse_answers(row[COLS["answers"]])

        image_path = str(row[COLS["image"]]).replace("./", "")
        image_path = os.path.join(TEST_ROOT, image_path)

        pred = predict_one(
            model=model,
            processor=processor,
            image_path=image_path,
            context=str(row[COLS["context"]]),
            question=str(row[COLS["question"]]),
            answers=answers,
            example_image_path=example_image_path,
            example_context=example_context,
            example_question=example_question,
            example_answers=example_answers,
            example_choice=example_choice,
        )

        predictions.append(pred)

    submission = pd.DataFrame({
        COLS["id"]: test_df[COLS["id"]],
        "label": predictions,
    })

    submission.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Second-stage count: {SECOND_STAGE_COUNT} / {len(test_df)}")
    print(f"Second-stage ratio: {SECOND_STAGE_COUNT / len(test_df):.2%}")
    print(f"Saved submission to: {OUTPUT_CSV_PATH}")

run_inference()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

100%|██████████| 8500/8500 [3:58:51<00:00,  1.69s/it]  

Second-stage count: 4125 / 8500
Second-stage ratio: 48.53%
Saved submission to: /content/drive/MyDrive/멀티모달 AI 챌린지/submission/submission.csv
